# 05 · Memory Management: References, Garbage Collection, `weakref`, Copying & Pickling

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 55 min

### 🎯 What you will learn
- How CPython manages memory: **reference counting** + the **cyclic garbage collector**
- The object lifecycle: `__new__` → `__init__` → … → `__del__` / finalizers
- `gc`: generations, thresholds, finding cycles
- **Weak references**: caches and observers that don't keep objects alive
- Object sizes: `sys.getsizeof` vs real (deep) size; list over-allocation; interning
- Memory leaks in Python and how to find them (`tracemalloc`)
- The **copy** (`__copy__`, `__deepcopy__`) and **pickle** (`__getstate__`, `__setstate__`, `__reduce__`) protocols

## 1. Reference counting

Every CPython object stores a count of references pointing to it. When the count drops to **0**, the object is freed **immediately**.

In [1]:
import sys

data = [1, 2, 3]
print(sys.getrefcount(data))      # +1 because getrefcount's own argument is a temporary reference
alias = data
container = {"key": data}
print(sys.getrefcount(data))
del alias
container.clear()
print(sys.getrefcount(data))

2
4
2


References are created by: assignment, putting the object in a container, passing it as an argument, closures, attributes... and removed by `del`, reassignment, going out of scope, removing from a container.

In [2]:
class Noisy:
    def __init__(self, name):
        self.name = name
        print(f"  + created {name}")
    def __del__(self):
        print(f"  - destroyed {self.name}")

def scope():
    n = Noisy("local")
    print("  leaving function")

scope()                        # destroyed as soon as the function returns
x = Noisy("x")
x = Noisy("y")                 # rebinding x drops the last reference to "x"
del x

  + created local
  leaving function
  - destroyed local
  + created x
  + created y
  - destroyed x
  - destroyed y


## 2. Reference cycles and the garbage collector

Reference counting can't free **cycles**: objects that refer to each other keep each other's count above zero. CPython's **cyclic GC** periodically finds and frees unreachable cycles.

In [3]:
import gc

class Node:
    def __init__(self, name):
        self.name, self.partner = name, None
    def __del__(self):
        print(f"  - collected {self.name}")

a, b = Node("A"), Node("B")
a.partner, b.partner = b, a              # a cycle: A ↔ B
del a, b                                 # no "collected" yet: the refcounts are still 1
print("after del, before gc")
print("unreachable objects found:", gc.collect())

after del, before gc
  - collected A
  - collected B
unreachable objects found: 8


### Generations

The GC groups objects into **generations**; most objects die young, so young generations are scanned most often.

In [4]:
print("thresholds:", gc.get_threshold())       # when automatic collections trigger
print("counts    :", gc.get_count())
print("enabled   :", gc.isenabled())
print(len(gc.get_objects()), "objects tracked by the GC right now")

thresholds: (2000, 10, 10)
counts    : (194, 0, 0)
enabled   : True
61078 objects tracked by the GC right now


`gc.disable()` is sometimes used in short-lived, allocation-heavy scripts for speed, but then cycles leak until `gc.collect()`.

## 3. Weak references

A **weak reference** points to an object **without** increasing its refcount, so the object can still be freed.

In [5]:
import weakref

class Image:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return f"Image({self.name!r})"

img = Image("cat.png")
ref = weakref.ref(img)
print(ref(), ref() is img)
del img
print(ref())                      # None: the object is gone

Image('cat.png') True
None


### A cache that doesn't cause memory leaks: `WeakValueDictionary`

In [6]:
cache = weakref.WeakValueDictionary()

def load(name):
    if (obj := cache.get(name)) is not None:
        print(f"  cache hit for {name}")
        return obj
    obj = Image(name)
    cache[name] = obj
    return obj

i1 = load("dog.png")
i2 = load("dog.png")
print(i1 is i2, dict(cache))
del i1, i2
print("after the last user is gone:", dict(cache))

  cache hit for dog.png
True {'dog.png': Image('dog.png')}
after the last user is gone: {}


Also: `WeakKeyDictionary` (attach data to objects you don't own), `WeakSet` (track live instances, observers), `weakref.WeakMethod` (callbacks to bound methods).

### `weakref.finalize`: reliable clean-up (better than `__del__`)

In [7]:
import tempfile, os

class TempWorkspace:
    def __init__(self):
        self.path = tempfile.mkdtemp()
        self._finalizer = weakref.finalize(self, os.rmdir, self.path)   # runs when the object dies (or at exit)
    def close(self):
        self._finalizer()                                              # or explicitly

ws = TempWorkspace()
p = ws.path
print(os.path.exists(p))
del ws
print(os.path.exists(p))

True
False


> ⚠️ Prefer `with` (context managers, Module 09) or `weakref.finalize` over `__del__`: `__del__` runs at unpredictable times, exceptions in it are ignored, and objects with it can complicate cycle collection.

## 4. Object sizes

In [8]:
for obj in [0, 1, 2**100, 1.5, "", "a", "hello world", b"", [], [1, 2, 3], (), {}, set(), object()]:
    print(f"{obj!r:<15} {sys.getsizeof(obj):>4} bytes")

0                 28 bytes
1                 28 bytes
1267650600228229401496703205376   40 bytes
1.5               24 bytes
''                41 bytes
'a'               42 bytes
'hello world'     52 bytes
b''               33 bytes
[]                56 bytes
[1, 2, 3]         88 bytes
()                48 bytes
{}                64 bytes
set()            216 bytes
<object object at 0x101cff510>   16 bytes


`sys.getsizeof` counts **only the object itself**, not what it refers to:

In [9]:
nested = [list(range(1000)) for _ in range(10)]
print(sys.getsizeof(nested), "bytes (just the outer list of 10 pointers)")

def deep_size(obj, seen=None):
    seen = set() if seen is None else seen
    if id(obj) in seen:
        return 0
    seen.add(id(obj))
    size = sys.getsizeof(obj)
    if isinstance(obj, dict):
        size += sum(deep_size(k, seen) + deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(deep_size(x, seen) for x in obj)
    elif hasattr(obj, "__dict__"):
        size += deep_size(vars(obj), seen)
    return size

print(deep_size(nested), "bytes (deep)")

184 bytes (just the outer list of 10 pointers)
108744 bytes (deep)


### List over-allocation

In [10]:
lst, last = [], None
for i in range(20):
    size = sys.getsizeof(lst)
    if size != last:
        print(f"len={len(lst):>2} → {size} bytes")
        last = size
    lst.append(i)

len= 0 → 56 bytes
len= 1 → 88 bytes
len= 5 → 120 bytes
len= 9 → 184 bytes
len=17 → 248 bytes


Lists reserve extra capacity so that `append` is amortised O(1).

### Interning and caching

In [11]:
import sys
largest = max(i for i in range(-10, 5000) if int(str(i)) is int(str(i)))
print(f"Python {sys.version_info[0]}.{sys.version_info[1]} caches ints from -5 up to {largest}")
a = 100_000; b = int("100000")
print(a is b)                       # outside the cache → two different objects
s1 = "hello_world"; s2 = "".join(["hello_", "world"])
print(s1 == s2, s1 is s2, sys.intern(s1) is sys.intern(s2))

Python 3.15 caches ints from -5 up to 1024
False
True False True


`sys.intern()` can save memory when you hold millions of repeated strings (e.g. dictionary keys parsed from files).

### Memory-saving techniques
- `__slots__` on classes with many instances (Module 06 · 07)
- generators instead of lists (Module 09 · 02)
- `array.array` / NumPy for large numeric data (Module 03 · 07)
- `weakref` caches; `del` large objects you no longer need; process data in chunks

## 5. Finding memory leaks

In Python, "leaks" usually mean objects that are **still referenced** by accident: global caches that only grow, listeners never removed, closures capturing big objects, unbounded `lru_cache`.

In [12]:
import tracemalloc

leaky_cache = []
def handle_request(i):
    leaky_cache.append(bytearray(10_000))      # bug: never cleared

tracemalloc.start()
snap1 = tracemalloc.take_snapshot()
for i in range(500):
    handle_request(i)
snap2 = tracemalloc.take_snapshot()
for stat in snap2.compare_to(snap1, "lineno")[:3]:
    print(stat)
tracemalloc.stop()
leaky_cache.clear()

/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/ipykernel_6172/1319615603.py:5: size=4934 KiB (+4934 KiB), count=1001 (+1001), average=5048 B
/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none/lib/python3.15/tracemalloc.py:560: size=328 B (+328 B), count=1 (+1), average=328 B
/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/uvpy/cpython-3.15.0-macos-aarch64-none/lib/python3.15/tracemalloc.py:423: size=328 B (+328 B), count=1 (+1), average=328 B


Other tools: `gc.get_referrers(obj)` (who keeps this alive?), `objgraph`, `memray` (Bloomberg's memory profiler), `pympler`.

## 6. The copy protocol

`copy.copy()` (shallow) and `copy.deepcopy()` (deep) can be customised:

In [13]:
import copy

class Document:
    def __init__(self, title, pages, cache=None):
        self.title, self.pages, self.cache = title, pages, cache or {}
    def __copy__(self):
        return Document(self.title, self.pages)                  # don't copy the cache
    def __deepcopy__(self, memo):
        return Document(copy.deepcopy(self.title, memo), copy.deepcopy(self.pages, memo))

d = Document("Guide", [["p1"], ["p2"]], cache={"render": "<html>"})
shallow, deep = copy.copy(d), copy.deepcopy(d)
d.pages[0].append("edited")
print(shallow.pages, shallow.cache)       # shares page lists, empty cache
print(deep.pages)                         # fully independent

[['p1', 'edited'], ['p2']] {}
[['p1'], ['p2']]


`memo` prevents infinite recursion on cyclic structures. 🆕 Python 3.13 added `copy.replace(obj, **changes)` (via `__replace__`), used by dataclasses, namedtuples and `datetime`.

## 7. The pickle protocol

`pickle` serialises object graphs. By default it saves `__dict__`; customise with `__getstate__` / `__setstate__`, or full control with `__reduce__`:

In [14]:
import pickle, threading

class Connection:
    def __init__(self, host):
        self.host = host
        self.lock = threading.Lock()          # locks can't be pickled!
        self.connected = True

    def __getstate__(self):
        state = self.__dict__.copy()
        del state["lock"]                     # drop unpicklable/transient parts
        state["connected"] = False
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self.lock = threading.Lock()          # recreate on load

c = Connection("db.example.com")
restored = pickle.loads(pickle.dumps(c))
print(restored.host, restored.connected, type(restored.lock).__name__)

class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __reduce__(self):                     # "to rebuild me, call Point(x, y)"
        return (Point, (self.x, self.y))

p = pickle.loads(pickle.dumps(Point(3, 4)))
print(p.x, p.y, pickle.HIGHEST_PROTOCOL)

db.example.com False lock
3 4 5


> ⚠️ Unpickling runs code (`__reduce__` can return *any* callable), so **never unpickle untrusted data**.

## 📝 Summary
- CPython frees objects when their refcount hits 0; the cyclic GC handles reference cycles.
- Use `weakref` for caches/observers that shouldn't keep objects alive; `weakref.finalize` for clean-up.
- `sys.getsizeof` is shallow; measure real usage with deep traversal or `tracemalloc`.
- Customise copying with `__copy__`/`__deepcopy__` and pickling with `__getstate__`/`__setstate__`/`__reduce__`.

## 🏋️ Practice
1. Create a parent/child tree where children keep a **weak** reference to their parent, and show that deleting the parent frees it without `gc.collect()`.
2. Use `tracemalloc` to find which line allocates the most memory in a function of your choice.
3. Make a class with an open file handle picklable by storing only the filename and reopening it in `__setstate__`.

<details><summary>Show solution for 1</summary>

```python
import weakref

class TreeNode:
    def __init__(self, name, parent=None):
        self.name = name
        self.children = []
        self._parent = weakref.ref(parent) if parent else None
        if parent:
            parent.children.append(self)
    @property
    def parent(self):
        return self._parent() if self._parent else None
    def __del__(self):
        print("freed", self.name)

root = TreeNode("root")
child = TreeNode("child", root)
print(child.parent.name)
del root          # freed immediately: no strong cycle
print(child.parent)
```
</details>